<a href="https://colab.research.google.com/github/ritiksharmasde/PROJECTS/blob/main/data_preprocessing_joshtalk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
import os
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive"

AUDIO_DIR = os.path.join(BASE_DIR, "audio")
TEXT_DIR = os.path.join(BASE_DIR, "text")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(BASE_DIR, "data.csv"))

print("Total rows:", len(df))


def fix_url(url):
    return str(url).replace(
        "joshtalks-data-collection/hq_data/hi",
        "upload_goai"
    ).strip()


def download(url, path):
    try:
        if os.path.exists(path):
            return True

        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            with open(path, "wb") as f:
                f.write(r.content)
            return True
    except:
        pass
    return False


# Counters
total = 0
success = 0
failed = 0


for _, row in tqdm(df.iterrows(), total=len(df)):

    if str(row["language"]).strip() != "hi":
        continue

    total += 1

    recording_id = str(row["recording_id"]).strip()

    # ✅ FIX URLs directly
    audio_url = fix_url(row["rec_url_gcp"])
    text_url = fix_url(row["transcription_url_gcp"])

    audio_path = os.path.join(AUDIO_DIR, f"{recording_id}.wav")
    text_path = os.path.join(TEXT_DIR, f"{recording_id}.json")

    audio_ok = download(audio_url, audio_path)
    text_ok = download(text_url, text_path)

    if audio_ok and text_ok:
        success += 1
    else:
        failed += 1


print("\n===== FINAL RESULT =====")
print(f"Total Hindi samples: {total}")
print(f"Successfully downloaded: {success}")
print(f"Failed: {failed}")
print(f"Success Rate: {(success/total)*100:.2f}%")

Mounted at /content/drive
Total rows: 104


100%|██████████| 104/104 [00:00<00:00, 336.24it/s]


===== FINAL RESULT =====
Total Hindi samples: 104
Successfully downloaded: 104
Failed: 0
Success Rate: 100.00%


In [2]:
# =========================
# 1. Setup
# =========================
import os
import json
import librosa
from tqdm import tqdm

BASE_DIR = "/content/drive/MyDrive"

AUDIO_DIR = os.path.join(BASE_DIR, "audio")
TEXT_DIR = os.path.join(BASE_DIR, "text")
OUTPUT_PATH = os.path.join(BASE_DIR, "train_segmented.jsonl")


# =========================
# 2. Clean Text
# =========================
def clean_text(text):
    text = str(text).strip()
    text = text.replace("\n", " ")

    fillers = ["जी जी जी", "जी जी", "हां हां", "हा हा", "अ ", "आ "]
    for f in fillers:
        text = text.replace(f, " ")

    text = text.replace("REDACTED", "")
    text = " ".join(text.split())

    return text


# =========================
# 3. Process Dataset (SEGMENT LEVEL)
# =========================
data = []

audio_files = os.listdir(AUDIO_DIR)

for file in tqdm(audio_files):

    if not file.endswith(".wav"):
        continue

    recording_id = file.replace(".wav", "")

    audio_path = os.path.join(AUDIO_DIR, file)
    text_path = os.path.join(TEXT_DIR, f"{recording_id}.json")

    if not os.path.exists(text_path):
        continue

    try:
        # Load full audio
        audio, sr = librosa.load(audio_path, sr=16000)

        with open(text_path, "r") as f:
            segments = json.load(f)

        # Iterate over segments
        for seg in segments:

            start = seg.get("start", None)
            end = seg.get("end", None)
            text = clean_text(seg.get("text", ""))

            # Skip bad segments
            if start is None or end is None:
                continue
            if len(text) < 3:
                continue

            start_sample = int(start * sr)
            end_sample = int(end * sr)

            segment_audio = audio[start_sample:end_sample]
            duration = len(segment_audio) / sr

            # Skip very short or very long clips
            if duration < 1 or duration > 30:
                continue

            sample = {
                "audio_filepath": audio_path,
                "text": text,
                "start": start,
                "end": end,
                "duration": duration,
                "language": "hi"
            }

            data.append(sample)

    except:
        continue


print("Total segmented samples:", len(data))


# =========================
# 4. Save JSONL
# =========================
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item in data:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

print("Saved file:", OUTPUT_PATH)

100%|██████████| 104/104 [02:32<00:00,  1.47s/it]

Total segmented samples: 4467
Saved file: /content/drive/MyDrive/train_segmented.jsonl


In [ ]:
# =========================================
# 1. INSTALL
# =========================================
!pip install -q datasets transformers accelerate evaluate jiwer librosa

# =========================================
# 2. IMPORTS
# =========================================
import torch
import librosa
import evaluate
from datasets import load_dataset
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    TrainingArguments,
    Trainer
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================
# 3. LOAD DATA
# =========================================
dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/train_segmented.jsonl"
)["train"]

print("Dataset size:", len(dataset))

# =========================================
# 4. MODEL
# =========================================
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

# 🔥 IMPORTANT (Hindi)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe"
)

model.generation_config.suppress_tokens = []

# =========================================
# 5. PREPROCESS
# =========================================
def prepare_dataset(batch):
    audio, sr = librosa.load(batch["audio_filepath"], sr=16000)

    start = int(batch["start"] * sr)
    end = int(batch["end"] * sr)
    audio = audio[start:end]

    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=2
)

# =========================================
# 6. SPLIT
# =========================================
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# =========================================
# 7. DATA COLLATOR (VERY IMPORTANT)
# =========================================
@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2Seq(processor)

# =========================================
# 8. TRAINING CONFIG (FIXED)
# =========================================
training_args = TrainingArguments(
    output_dir="/content/whisper-small-hi",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=200,  # 🔥 faster
    eval_strategy="steps",   # ✅ FIXED
    save_steps=100,
    eval_steps=100,
    logging_steps=50,
    fp16=True,
    save_total_limit=2,
    report_to="none"
)

# =========================================
# 9. TRAIN
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

# =========================================
# 10. LOAD FLEURS
# =========================================
fleurs = load_dataset("google/fleurs", "hi_in", split="test")

wer_metric = evaluate.load("wer")

# =========================================
# 11. EVAL FUNCTION
# =========================================
def evaluate_model(model, dataset, num_samples=20):  # 🔥 faster
    model.eval()

    preds, refs = [], []

    for i in range(num_samples):
        sample = dataset[i]
        audio = sample["audio"]["array"]

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            pred_ids = model.generate(inputs.input_features)

        pred_text = processor.batch_decode(
            pred_ids, skip_special_tokens=True
        )[0]

        preds.append(pred_text)
        refs.append(sample["transcription"])

    return wer_metric.compute(predictions=preds, references=refs)

# =========================================
# 12. BASELINE
# =========================================
baseline_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

baseline_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi", task="transcribe"
)

baseline_wer = evaluate_model(baseline_model, fleurs)
print("Baseline WER:", baseline_wer)

# =========================================
# 13. FINETUNED
# =========================================
fine_tuned_wer = evaluate_model(trainer.model, fleurs)
print("Fine-tuned WER:", fine_tuned_wer)

# =========================================
# 14. RESULT
# =========================================
print("\n===== FINAL RESULTS =====")
print(f"Baseline WER    : {baseline_wer:.4f}")
print(f"Fine-tuned WER  : {fine_tuned_wer:.4f}")


Generating train split: 0 examples [00:00, ? examples/s]

Dataset size: 4467


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Map (num_proc=2):   0%|          | 0/4467 [00:00<?, ? examples/s]

In [ ]:
!mkdir /content/drive/MyDrive/local_scripts
!mv /content/drive/MyDrive/fleurs.py /content/drive/MyDrive/local_scripts/

In [ ]:
!ls /content/drive/MyDrive/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!ls /content/

In [1]:
# =========================================
# 1️⃣ INSTALL REQUIRED PACKAGES
# =========================================
!pip install -q datasets transformers accelerate evaluate jiwer librosa

# =========================================
# 2️⃣ IMPORTS
# =========================================
import torch
import librosa
import evaluate
from datasets import load_dataset
from dataclasses import dataclass
from typing import Any
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================
# 3️⃣ MOUNT GOOGLE DRIVE
# =========================================
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive"
AUDIO_DIR = os.path.join(BASE_DIR, "audio")
TEXT_DIR = os.path.join(BASE_DIR, "text")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)

# =========================================
# 4️⃣ LOAD YOUR DATASET
# =========================================
dataset = load_dataset(
    "json",
    data_files=os.path.join(BASE_DIR, "train_segmented.jsonl")
)["train"]

print("Full dataset size:", len(dataset))

# =========================================
# 5️⃣ DEFINE PROCESSING FUNCTION
# =========================================
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

def prepare_dataset(batch):
    audio, sr = librosa.load(batch["audio_filepath"], sr=16000)
    start = int(batch["start"] * sr)
    end = int(batch["end"] * sr)
    audio = audio[start:end]

    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

# =========================================
# 6️⃣ QUICK TEST: SMALL SUBSET
# =========================================
small_dataset = dataset.select(range(2))

small_dataset = small_dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=1
)
print("Small subset preprocessing done ✅")

# =========================================
# 7️⃣ LOAD WER METRIC
# =========================================
wer_metric = evaluate.load("wer")

# =========================================
# 8️⃣ PREPARE FOR FULL FINE-TUNING
# =========================================
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2Seq(processor)

# =========================================
# 9️⃣ LOAD & TRAIN MODEL
# =========================================
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(DEVICE)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")
model.generation_config.suppress_tokens = []

training_args = TrainingArguments(
    output_dir="/content/whisper-small-hi",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_steps=2,
    eval_steps=1,
    save_steps=1,
    logging_steps=1,
    save_total_limit=2,
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_dataset,
    eval_dataset=small_dataset,
    data_collator=data_collator
)

trainer.train()
print("Quick test fine-tuning done ✅")

# =========================================
# 🔟 PREPROCESS FULL EVAL DATASET
# =========================================
print("Preprocessing eval dataset...")

eval_dataset_processed = eval_dataset.map(
    prepare_dataset,
    remove_columns=eval_dataset.column_names,
    num_proc=1
)

print(f"Eval dataset size: {len(eval_dataset_processed)}")

# =========================================
# 1️⃣1️⃣ EVALUATION FUNCTION
# =========================================
def evaluate_on_own_data(model, eval_data, num_samples=20):
    model.eval()
    preds, refs = [], []

    for i in range(min(num_samples, len(eval_data))):
        sample = eval_data[i]

        inputs = torch.tensor([sample["input_features"]]).to(DEVICE)

        with torch.no_grad():
            pred_ids = model.generate(inputs)

        pred_text = processor.batch_decode(
            pred_ids, skip_special_tokens=True
        )[0]

        ref_text = processor.tokenizer.decode(
            [t for t in sample["labels"] if t != -100],
            skip_special_tokens=True
        )

        preds.append(pred_text)
        refs.append(ref_text)

    return wer_metric.compute(predictions=preds, references=refs)

# =========================================
# 1️⃣2️⃣ BASELINE MODEL
# =========================================
print("Evaluating baseline model...")

baseline_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

baseline_model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi", task="transcribe"
)

baseline_wer = evaluate_on_own_data(baseline_model, eval_dataset_processed)
print(f"Baseline WER: {baseline_wer:.4f}")

# =========================================
# 1️⃣3️⃣ FINE-TUNED MODEL
# =========================================
print("Evaluating fine-tuned model...")

fine_tuned_wer = evaluate_on_own_data(trainer.model, eval_dataset_processed)
print(f"Fine-tuned WER: {fine_tuned_wer:.4f}")

# =========================================
# 1️⃣4️⃣ FINAL RESULTS
# =========================================
print("\n===== FINAL RESULTS =====")
print(f"Baseline WER    : {baseline_wer:.4f}")
print(f"Fine-tuned WER  : {fine_tuned_wer:.4f}")
print(f"Improvement     : {((baseline_wer - fine_tuned_wer) / baseline_wer) * 100:.2f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.1 MB/s eta 0:00:00
Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Full dataset size: 4467


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/447 [00:00<?, ? examples/s]

Eval dataset size: 447
Evaluating baseline model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

Baseline WER: 2.1369
Evaluating fine-tuned model...
Fine-tuned WER: 2.4223

===== FINAL RESULTS =====
Baseline WER    : 2.1369
Fine-tuned WER  : 2.4223
Improvement     : -13.36%


In [2]:
# =========================================
# 1️⃣ INSTALL REQUIRED PACKAGES
# =========================================
!pip install -q datasets transformers accelerate evaluate jiwer librosa

# =========================================
# 2️⃣ IMPORTS
# =========================================
import torch
import librosa
import evaluate
import numpy as np
from datasets import load_dataset
from dataclasses import dataclass
from typing import Any
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# =========================================
# 3️⃣ MOUNT GOOGLE DRIVE
# =========================================
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive"
AUDIO_DIR = os.path.join(BASE_DIR, "audio (1)")
TEXT_DIR = os.path.join(BASE_DIR, "text (1)")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)

# =========================================
# 4️⃣ LOAD FULL DATASET
# =========================================
dataset = load_dataset(
    "json",
    data_files=os.path.join(BASE_DIR, "train_segmented.jsonl")
)["train"]

print("Full dataset size:", len(dataset))

# =========================================
# 5️⃣ DEFINE PROCESSING FUNCTION
# =========================================
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

def prepare_dataset(batch):
    audio, sr = librosa.load(batch["audio_filepath"], sr=16000)
    start = int(batch["start"] * sr)
    end = int(batch["end"] * sr)
    audio = audio[start:end]

    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

# =========================================
# 6️⃣ PREPROCESS FULL DATASET
# =========================================
print("Preprocessing full dataset... (this will take time)")

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=2
)

print("Full preprocessing done ✅")

# =========================================
# 7️⃣ SPLIT TRAIN / EVAL
# =========================================
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")

# =========================================
# 8️⃣ DATA COLLATOR
# =========================================
@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2Seq(processor)

# =========================================
# 9️⃣ LOAD MODEL
# =========================================
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi", task="transcribe"
)
model.generation_config.suppress_tokens = []

# =========================================
# 🔟 TRAINING CONFIG — 3 EPOCHS
# =========================================
training_args = TrainingArguments(
    output_dir="/content/whisper-small-hi",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=3,           # ✅ 3 epochs
    eval_strategy="epoch",        # ✅ evaluate after each epoch
    save_strategy="epoch",        # ✅ save after each epoch
    logging_steps=50,
    fp16=True,
    save_total_limit=2,
    load_best_model_at_end=True,  # ✅ keeps best checkpoint
    report_to="none"
)

# =========================================
# 1️⃣1️⃣ TRAIN
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

print("Starting full training (3 epochs)...")
trainer.train()
print("Full training done ✅")

# =========================================
# 1️⃣2️⃣ WER METRIC
# =========================================
wer_metric = evaluate.load("wer")

# =========================================
# 1️⃣3️⃣ EVALUATION FUNCTION
# =========================================
def evaluate_on_own_data(model, eval_data, num_samples=100):
    model.eval()
    preds, refs = [], []

    for i in range(min(num_samples, len(eval_data))):
        sample = eval_data[i]

        # ✅ Safe conversion
        features = np.array(sample["input_features"], dtype=np.float32)
        inputs = torch.tensor(features).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_ids = model.generate(inputs)

        pred_text = processor.batch_decode(
            pred_ids, skip_special_tokens=True
        )[0]

        ref_text = processor.tokenizer.decode(
            [t for t in sample["labels"] if t != -100],
            skip_special_tokens=True
        )

        preds.append(pred_text)
        refs.append(ref_text)

        if (i + 1) % 10 == 0:
            print(f"Evaluated {i+1}/{num_samples} samples...")

    return wer_metric.compute(predictions=preds, references=refs)

# =========================================
# 1️⃣4️⃣ BASELINE MODEL
# =========================================
print("\nEvaluating baseline model...")

baseline_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

baseline_model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi", task="transcribe"
)

baseline_wer = evaluate_on_own_data(baseline_model, eval_dataset)
print(f"Baseline WER: {baseline_wer:.4f}")

# =========================================
# 1️⃣5️⃣ FINE-TUNED MODEL
# =========================================
print("\nEvaluating fine-tuned model...")

fine_tuned_wer = evaluate_on_own_data(trainer.model, eval_dataset)
print(f"Fine-tuned WER: {fine_tuned_wer:.4f}")

# =========================================
# 1️⃣6️⃣ FINAL RESULTS
# =========================================
print("\n===== FINAL RESULTS =====")
print(f"Baseline WER    : {baseline_wer:.4f}")
print(f"Fine-tuned WER  : {fine_tuned_wer:.4f}")
print(f"Improvement     : {((baseline_wer - fine_tuned_wer) / baseline_wer) * 100:.2f}%")

# =========================================
# 💾 SAVE MODEL TO GOOGLE DRIVE
# =========================================
save_path = os.path.join(BASE_DIR, "whisper-small-hi-final")

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

Using device: cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full dataset size: 4467


Parameter 'function'=<function prepare_dataset at 0x7fc7ffa9b2e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Preprocessing full dataset... (this will take time)


Map (num_proc=2):   0%|          | 0/4467 [00:00<?, ? examples/s]

TimeoutError: 